<div style="background-color: #1e1e2e; padding: 20px; border-radius: 10px; border-left: 5px solid #fab387;">
    <h1 style="color: #cba6f7; margin-top: 0; font-family: sans-serif;">⚡ Model Training: ResNet-18 Baseline</h1>
    <h3 style="color: #bac2de; margin-bottom: 0; font-family: sans-serif; font-weight: normal;">Lung Cancer Detection from CT Scans Dataset</h3>
    <hr style="border: 1px solid #313244; margin-top: 15px; margin-bottom: 15px;">
    <p style="color: #a6adc8; margin-bottom: 0; font-family: sans-serif;">
        This notebook uses high-performance deep learning with PyTorch and a ResNet-18 architecture to establish a competitive baseline classifier. It evaluates selection accuracy and cross-entropy loss on the train and validation splits.
    </p>
</div>


### 📦 1. Importing Libraries
We begin by importing required libraries, primarily from the `torch` ecosystem for model building, data loading, and preprocessing.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torchvision import models
import os
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### ⚙️ 2. Hyperparameters & Configuration
Defining the core hyperparameters that will control the training process, such as epochs, batch size, and learning rate.


In [ ]:
num_epochs = 10
batch_size = 64  # Increased for NVIDIA P100 (16GB VRAM)
learning_rate = 0.0001
num_classes = 2


### 🖼️ 3. Data Transformations & Loading
We define our image transformations (resizing and normalization). We then load the dataset from the `Dataset` folder, splitting it sequentially into `80%` training data and `20%` validation data.


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(),         
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) 
])

dataset = ImageFolder(root="Dataset", transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)


### 🏗️ 4. Model Architecture Selection
We leverage a pre-trained `ResNet-18` architecture from `torchvision` to take advantage of transfer learning. We modify the final fully connected (`fc`) layer to match our `num_classes` (Positive vs Negative).


In [ ]:
from torchvision.models import ResNet18_Weights
model = models.resnet18(weights = ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, num_classes)  
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


### 🏃 5. Training Loop
The `train_model` function encompasses both the training loop (where the model learns to minimize loss) and the validation phase (where we check the model's accuracy on unseen data).


In [ ]:
def train_model():
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0

        train_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", unit="batch")
        for images, labels in train_progress:
            images, labels = images.to(device), labels.to(device)
            labels = labels.long()

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()

            train_progress.set_postfix(loss=(running_loss / len(train_loader)))

        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}")

        # Validation
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        accuracy = 100 * correct / total
        print(f"Validation Accuracy: {accuracy:.2f}%")

train_model()


### 💾 6. Saving the Model Weights
Once training concludes, we save the learned weights to our `Model` directory. This allows us to load the model in our application (`app.ipynb`) without needing to retrain.


In [ ]:
from safetensors.torch import save_model
save_model(model, "Model/lung_cancer_cnn_v1.safetensors")


### 🎉 Conclusion
The ResNet-18 model was successfully trained on the lung CT scan dataset. By utilizing transfer learning and optimizing for our specific dataset, we have established a strong baseline model capable of distinguishing between positive and negative cases. The model weights are saved and ready for deployment.
